# Telecom Silver Layer (Data Cleaning)
Adhering to strict Medallion Architecture: Reading ALL raw data (Streaming Delta & Batch CSVs) from Bronze, cleaning, and writing to Silver Delta tables.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, to_timestamp, when, current_timestamp, to_date, year, month, date_format

print("Starting Spark Session...")
spark = SparkSession.builder \
    .appName("Telecom_Silver") \
    .config("spark.jars.packages", "io.delta:delta-spark_2.12:3.2.0,org.apache.hadoop:hadoop-aws:3.3.4") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "admin") \
    .config("spark.hadoop.fs.s3a.secret.key", "password") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .getOrCreate()

print("Spark Session Ready!")

In [ ]:
# 1. Read & Clean Streaming CDRs (From Bronze Delta Table written by Kafka Stream)
print("Processing CDRs from Bronze...")
df_raw = spark.read.format("delta").load("s3://bronze/telecom_cdrs_raw")

df_clean = df_raw.withColumn("timestamp", to_timestamp(col("timestamp"), "yyyy-MM-dd HH:mm:ss")) \
                 .withColumn("date_id", to_date(col("timestamp"))) \
                 .withColumn("duration_seconds", col("duration_seconds").cast("integer")) \
                 .withColumn("data_consumed_mb", col("data_consumed_mb").cast("integer")) \
                 .withColumn("latency_ms", col("latency_ms").cast("integer")) \
                 .withColumn("signal_strength_dbm", col("signal_strength_dbm").cast("integer"))

# Business Logic: Voice calls shouldn't consume data, and Data sessions shouldn't have duration
df_clean = df_clean.withColumn("data_consumed_mb", when(col("event_type") == "voice", 0).otherwise(col("data_consumed_mb"))) \
                   .withColumn("duration_seconds", when(col("event_type") == "data", 0).otherwise(col("duration_seconds")))

# Data Quality (Quarantine)
df_validated = df_clean.withColumn("is_valid", when(col("call_id").isNull() | col("customer_number").isNull(), False).otherwise(True))
df_good = df_validated.filter(col("is_valid") == True).drop("is_valid")
df_bad = df_validated.filter(col("is_valid") == False).drop("is_valid").withColumn("quarantine_ts", current_timestamp())

df_good.write.format("delta").mode("append").option("mergeSchema", "true").save("s3://silver/telecom_cdrs")
if df_bad.count() > 0: df_bad.write.format("delta").mode("append").option("mergeSchema", "true").save("s3://silver/quarantine_cdrs")


In [ ]:
# 2. Read Batch CRM Dimensions (From Bronze CSVs)
print("Processing CRM Data from Bronze...")

# Customers
spark.read.format("csv").option("header", "true").load("s3://bronze/customers.csv") \
    .write.format("delta").mode("overwrite").save("s3://silver/customers_dim_seed")

# Towers
spark.read.format("csv").option("header", "true").load("s3://bronze/towers.csv") \
    .write.format("delta").mode("overwrite").save("s3://silver/towers_dim_seed")

# Plans
spark.read.format("csv").option("header", "true").load("s3://bronze/plans.csv") \
    .write.format("delta").mode("overwrite").save("s3://silver/seed_plan")

# Events
spark.read.format("csv").option("header", "true").load("s3://bronze/events.csv") \
    .write.format("delta").mode("overwrite").save("s3://silver/seed_event")

print("CRM Data processed to Silver.")

In [ ]:
# 3. Generate Date Dimension (Algorithmic, no Bronze source needed)
print("Generating Date Dimension...")
df_date = spark.sql("SELECT explode(sequence(to_date('2020-01-01'), to_date('2030-12-31'), interval 1 day)) as Date")
df_dim_date = df_date.withColumn("date_id", col("Date")) \
                     .withColumn("year", year(col("Date"))) \
                     .withColumn("month", month(col("Date"))) \
                     .withColumn("day_name", date_format(col("Date"), "EEEE"))
df_dim_date.write.format("delta").mode("overwrite").save("s3://silver/seed_date")

print("Silver Layer Complete!")